# DL Spring 2026 — Text-to-SVG | Rahil | Windows RTX 5060
**NYU Tandon | CS-GY 9223 / ECE-GY 7123**

AI Tooling Disclosure: Claude (Anthropic) used for code scaffolding and debugging.

In [1]:
# Cell 1 — Install
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'datasets', 'trl==0.24.0', 'peft', 'accelerate', 'bitsandbytes', 'lxml'], check=True)
print('All packages ready.')

All packages ready.


In [2]:
# Cell 2 — Imports & Seeds
import os, re, time, random, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('CUDA not available! Check GPU drivers.')

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


In [3]:
# Cell 3 — Config
# UPDATE these paths if your files are elsewhere
DATA_DIR = r'C:\Users\rahil\Downloads\dl-spring-2026-svg-generation'

CONFIG = {
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'train_path': os.path.join(DATA_DIR, 'train.csv'),
    'test_path':  os.path.join(DATA_DIR, 'test.csv'),
    'output_dir': r'C:\Users\rahil\svg_lora_output',
    'submission_path': os.path.join(DATA_DIR, 'submission.csv'),

    # LoRA
    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,

    # Training — tuned for 8GB VRAM RTX 5060
    'max_seq_length': 1024,
    'max_train_samples': 10000,
    'num_train_epochs': 2,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'learning_rate': 2e-4,
    'weight_decay': 0.01,
    'max_grad_norm': 0.3,
    'warmup_steps': 100,
    'logging_steps': 25,
    'eval_steps': 200,
    'save_steps': 400,
    'eval_size': 0.02,
    'min_svg_length': 50,
    'max_svg_length': 16000,
    'min_prompt_length': 5,

    # Inference
    'max_new_tokens': 600,
    'inference_batch_size': 2,
    'temperature': 0.3,
    'repetition_penalty': 1.1,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Config ready.')
print(f"Train: {CONFIG['train_path']}")
print(f"Test:  {CONFIG['test_path']}")
print(f"train.csv exists: {os.path.exists(CONFIG['train_path'])}")
print(f"test.csv exists:  {os.path.exists(CONFIG['test_path'])}")

Config ready.
Train: C:\Users\rahil\Downloads\dl-spring-2026-svg-generation\train.csv
Test:  C:\Users\rahil\Downloads\dl-spring-2026-svg-generation\test.csv
train.csv exists: True
test.csv exists:  True


In [4]:
# Cell 4 — SVG Utilities
ET.register_namespace('', 'http://www.w3.org/2000/svg')
SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "Output ONLY a single valid <svg> element. "
    "Start your response directly with <svg. "
    "Use width='256' height='256' viewBox='0 0 256 256'. "
    "No explanation, no markdown, just SVG code."
)

def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or len(svg_text) < 10:
        return False
    if not svg_text.strip().lower().startswith('<svg'):
        return False
    if len(svg_text) > 16000:
        return False
    for f in ['<script', 'javascript:', 'onload=', 'onclick=', 'onerror=']:
        if f.lower() in svg_text.lower():
            return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag == 'svg' or root.tag.endswith('}svg')
    except ET.ParseError:
        return False

def clean_svg(svg_text: str) -> str:
    """Normalize dimensions and remove duplicate attributes via re-serialization."""
    if not svg_text or not isinstance(svg_text, str):
        return ''
    try:
        root = ET.fromstring(svg_text)
        root.set('width', '256')
        root.set('height', '256')
        if 'viewBox' not in root.attrib:
            root.set('viewBox', '0 0 256 256')
        if 'xmlns' not in root.attrib:
            root.set('xmlns', 'http://www.w3.org/2000/svg')
        return ET.tostring(root, encoding='unicode')
    except Exception:
        return ''

def fallback_svg() -> str:
    return ('<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
            '<rect x="0" y="0" width="256" height="256" fill="white"/>'
            '<circle cx="128" cy="128" r="80" fill="#333333"/>'
            '</svg>')

def count_paths(svg_text: str) -> int:
    return len(re.findall(r'<path', svg_text, re.IGNORECASE))

print('SVG utilities ready.')

SVG utilities ready.


In [5]:
# Cell 5 — Load & Clean Training Data
print('Loading training data...')
train_df = pd.read_csv(CONFIG['train_path'])
print(f'Raw rows: {len(train_df):,}')

train_df = train_df.dropna(subset=['prompt', 'svg'])

# Filter by length only — do NOT run clean_svg on training data
# clean_svg fails on many valid training SVGs due to namespace variants
train_df = train_df[
    (train_df['svg'].str.len() >= CONFIG['min_svg_length']) &
    (train_df['svg'].str.len() <= CONFIG['max_svg_length']) &
    (train_df['prompt'].str.len() >= CONFIG['min_prompt_length'])
].reset_index(drop=True)
print(f'After length filter: {len(train_df):,}')

# Lenient validity gate
valid_mask = train_df['svg'].apply(is_valid_svg)
train_df = train_df[valid_mask].reset_index(drop=True)
print(f'After validity gate: {len(train_df):,}')

path_mask = train_df['svg'].apply(count_paths) <= 256
train_df = train_df[path_mask].reset_index(drop=True)
print(f'After path filter: {len(train_df):,}')

if CONFIG['max_train_samples'] and len(train_df) > CONFIG['max_train_samples']:
    train_df = train_df.sample(n=CONFIG['max_train_samples'], random_state=SEED).reset_index(drop=True)
    print(f'Subsampled to: {len(train_df):,}')

svg_len = train_df['svg'].str.len()
print(f'SVG length — min: {svg_len.min()}, mean: {svg_len.mean():.0f}, max: {svg_len.max()}')

Loading training data...
Raw rows: 50,000
After length filter: 50,000
After validity gate: 50,000
After path filter: 49,999
Subsampled to: 10,000
SVG length — min: 95, mean: 2530, max: 15038


In [6]:
# Cell 6 — Format for SFT
def format_sample(prompt: str, svg: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{svg}<|im_end|>'
    )

train_df['text'] = train_df.apply(lambda r: format_sample(r['prompt'], r['svg']), axis=1)

from datasets import Dataset

eval_n = max(100, int(len(train_df) * CONFIG['eval_size']))
eval_df = train_df.sample(n=eval_n, random_state=SEED)
train_split_df = train_df.drop(eval_df.index).reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

train_ds = Dataset.from_pandas(train_split_df[['text']])
eval_ds  = Dataset.from_pandas(eval_df[['text']])
print(f'Train: {len(train_ds):,} | Eval: {len(eval_ds):,}')
print(train_ds[0]['text'][:300])

C:\Users\rahil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 9,800 | Eval: 200
<|im_start|>system
You are an SVG code generator. Output ONLY a single valid <svg> element. Start your response directly with <svg. Use width='256' height='256' viewBox='0 0 256 256'. No explanation, no markdown, just SVG code.<|im_end|>
<|im_start|>user
A black icon featuring a vertical rectangle w


In [7]:
# Cell 7 — Load Model + QLoRA
# RTX 5060: fp16, single GPU, bitsandbytes 4-bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {CONFIG['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # right padding for TRAINING

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map={'': 0},           # single GPU — avoids DataParallel dtype crash
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading Qwen/Qwen2.5-Coder-1.5B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
W0327 09:20:08.750000 4604 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/338 [00:00<01:11,  4.69it/s]C:\Users\rahil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 234.53it/s]


trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [8]:
# Cell 8 — Train (with resume + sleep disabled)
import os, subprocess
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Disable Windows sleep during training
subprocess.run(['powercfg', '/change', 'standby-timeout-ac', '0'], capture_output=True)
subprocess.run(['powercfg', '/change', 'standby-timeout-dc', '0'], capture_output=True)
print('Sleep disabled.')

import torch
torch.cuda.empty_cache()

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_steps=CONFIG['warmup_steps'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    lr_scheduler_type='cosine',
    bf16=False,
    fp16=True,
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    report_to='none',
    seed=SEED,
    max_length=CONFIG['max_seq_length'],
    dataset_text_field='text',
    packing=True,
    resume_from_checkpoint=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
)

print(f'Effective batch size: {CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]}')
print('Resuming training from last checkpoint...')
t0 = time.time()
trainer.train()
print(f'Done in {(time.time()-t0)/60:.1f} min')

adapter_path = os.path.join(CONFIG['output_dir'], 'final_adapter')
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved to: {adapter_path}')

Sleep disabled.


[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-

Effective batch size: 16
Resuming training from last checkpoint...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.418014,0.426870,0.426484,3243503.000000,0.856795
400,0.359494,0.395792,0.405020,6484585.000000,0.867374
600,0.332525,0.378800,0.368776,9711458.000000,0.873160
800,0.323675,0.370121,0.363196,12954321.000000,0.875222
1000,0.313980,0.364242,0.358813,16195733.000000,0.877341
1116,0.336357,0.363761,0.358871,18061744.000000,0.877410


C:\Users\rahil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-Coder-1.5B-Instruct.
  warnings.warn(
C:\Users\rahil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-Coder-1.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Done in 717.3 min
Adapter saved to: C:\Users\rahil\svg_lora_output\final_adapter


In [18]:
# Must define clean_and_validate before generate_batch uses it
import xml.etree.ElementTree as ET

def clean_and_validate(svg_text):
    if not svg_text:
        return fallback_svg()
    try:
        ET.register_namespace('', 'http://www.w3.org/2000/svg')
        root = ET.fromstring(svg_text)
        if not (root.tag == 'svg' or root.tag.endswith('}svg')):
            return fallback_svg()
        root.set('width', '256')
        root.set('height', '256')
        if 'viewBox' not in root.attrib:
            root.set('viewBox', '0 0 256 256')
        return ET.tostring(root, encoding='unicode')
    except:
        return fallback_svg()

def generate_batch(prompts: list) -> list:
    formatted = [build_inference_prompt(p) for p in prompts]
    inputs = tokenizer(
        formatted,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True,
            temperature=CONFIG['temperature'],
            repetition_penalty=CONFIG['repetition_penalty'],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    results = []
    for i, ids in enumerate(output_ids):
        new_ids = ids[inputs['input_ids'].shape[1]:]
        decoded = tokenizer.decode(new_ids, skip_special_tokens=True)
        svg = extract_svg(decoded)
        if svg:
            # Try to clean — if cleaning fails keep original
            cleaned = clean_and_validate(svg)
            if cleaned and cleaned != fallback_svg():
                svg = cleaned
            # Accept if it starts with <svg regardless of namespace
            if svg.strip().lower().startswith('<svg'):
                results.append(svg)
                continue
        results.append(fallback_svg())
    return results

In [16]:
# Cell 10 — Sanity Check
# STOP after this cell and check RAW OUTPUT
# If it starts with <svg — proceed to full inference
# If it is garbage — DO NOT run full inference, training did not work
test_prompt = 'A simple red circle on a white background'
inp = tokenizer(build_inference_prompt(test_prompt), return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(
        input_ids=inp['input_ids'],
        attention_mask=inp['attention_mask'],
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
decoded = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
print('RAW OUTPUT:')
print(decoded[:400])
print(f'\nStarts with <svg: {decoded.strip().lower().startswith("<svg")}')
print(f'Extracted SVG valid: {is_valid_svg(extract_svg(decoded))}')

C:\Users\rahil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


RAW OUTPUT:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#FF4757" fill-opacity="1.0"  filling="0" d="M99.80499267578125 199.99999237060547 C44.76599884033203 199.99999237060547 0.0 155.23400115966797 0.0 100.19599914550781

Starts with <svg: True
Extracted SVG valid: False


In [20]:
# Fix OOM during inference — reduce batch size
CONFIG['inference_batch_size'] = 4
torch.cuda.empty_cache()
print('Batch size reduced to 4, cache cleared.')

Batch size reduced to 4, cache cleared.


In [21]:
# Cell 11 — Full Inference (only run if sanity check passed)
test_df = pd.read_csv(CONFIG['test_path'])
print(f'Test rows: {len(test_df)}')

all_svgs = []
invalid_count = 0
t0 = time.time()

for i in range(0, len(test_df), CONFIG['inference_batch_size']):
    batch = test_df['prompt'].tolist()[i:i + CONFIG['inference_batch_size']]
    svgs = generate_batch(batch)
    all_svgs.extend(svgs)
    invalid_count += sum(1 for s in svgs if s == fallback_svg())
    elapsed = (time.time() - t0) / 60
    print(f'  [{i+len(batch)}/1000] {elapsed:.1f}min | fallbacks: {invalid_count}')

print(f'\nDone. Fallbacks: {invalid_count}/1000')

Test rows: 1000
  [4/1000] 1.3min | fallbacks: 0
  [8/1000] 2.6min | fallbacks: 0
  [12/1000] 3.9min | fallbacks: 0
  [16/1000] 5.0min | fallbacks: 0
  [20/1000] 6.1min | fallbacks: 0
  [24/1000] 7.2min | fallbacks: 0
  [28/1000] 8.4min | fallbacks: 0
  [32/1000] 9.5min | fallbacks: 0
  [36/1000] 10.6min | fallbacks: 0
  [40/1000] 11.7min | fallbacks: 0
  [44/1000] 12.8min | fallbacks: 0
  [48/1000] 14.0min | fallbacks: 0
  [52/1000] 15.1min | fallbacks: 0
  [56/1000] 16.2min | fallbacks: 0
  [60/1000] 17.3min | fallbacks: 0
  [64/1000] 18.4min | fallbacks: 0
  [68/1000] 19.6min | fallbacks: 0
  [72/1000] 20.7min | fallbacks: 0
  [76/1000] 21.8min | fallbacks: 0
  [80/1000] 22.9min | fallbacks: 0
  [84/1000] 24.0min | fallbacks: 0
  [88/1000] 25.2min | fallbacks: 0
  [92/1000] 26.3min | fallbacks: 0
  [96/1000] 27.4min | fallbacks: 0
  [100/1000] 28.5min | fallbacks: 0
  [104/1000] 29.6min | fallbacks: 0
  [108/1000] 30.8min | fallbacks: 0
  [112/1000] 32.0min | fallbacks: 0
  [116/100

In [22]:
# Cell 12 : Build & Save Submission
import xml.etree.ElementTree as ET

def clean_and_validate(svg_text):
    if not svg_text:
        return fallback_svg()
    try:
        ET.register_namespace('', 'http://www.w3.org/2000/svg')
        root = ET.fromstring(svg_text)
        # Accept any namespace variant
        if not (root.tag == 'svg' or root.tag.endswith('}svg')):
            return fallback_svg()
        root.set('width', '256')
        root.set('height', '256')
        if 'viewBox' not in root.attrib:
            root.set('viewBox', '0 0 256 256')
        cleaned = ET.tostring(root, encoding='unicode')
        return cleaned
    except:
        return fallback_svg()

submission_df = pd.DataFrame({'id': test_df['id'], 'svg': all_svgs})
submission_df['svg'] = submission_df['svg'].apply(clean_and_validate)

# Count real SVGs vs fallbacks
fallback = fallback_svg()
real_count = sum(1 for s in submission_df['svg'] if s != fallback)
print(f'Real SVGs: {real_count}/1000')
print(f'Fallbacks: {1000 - real_count}/1000')

validity = submission_df['svg'].apply(lambda s: s != fallback and s.startswith('<svg'))
print(f'Valid: {validity.sum()}/1000 ({validity.mean()*100:.1f}%)')

submission_df.to_csv(CONFIG['submission_path'], index=False)
print(f"Saved: {CONFIG['submission_path']}")
print('Upload to Kaggle!')

Real SVGs: 129/1000
Fallbacks: 871/1000
Valid: 129/1000 (12.9%)
Saved: C:\Users\rahil\Downloads\dl-spring-2026-svg-generation\submission.csv
Upload to Kaggle!


In [ ]:
# Cell 13 — Run Summary
print('=' * 60)
print('RUN SUMMARY')
print('=' * 60)
print(f"Model:            {CONFIG['model_name']}")
print(f"LoRA r/alpha:     {CONFIG['lora_r']}/{CONFIG['lora_alpha']}")
print(f"Train samples:    {len(train_ds):,}")
print(f"Epochs:           {CONFIG['num_train_epochs']}")
print(f"LR:               {CONFIG['learning_rate']}")
print(f"Batch (eff):      {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"Max seq length:   {CONFIG['max_seq_length']}")
print(f"Valid SVG rate:   {validity.mean()*100:.1f}%")
print(f"Fallbacks:        {invalid_count}/1000")
print(f"Seed:             {SEED}")
print('=' * 60)
print('AI Tooling: Claude (Anthropic) used for code scaffolding and debugging.')

RUN SUMMARY
Model:            Qwen/Qwen2.5-Coder-1.5B-Instruct
LoRA r/alpha:     32/64
Train samples:    9,800
Epochs:           2
LR:               0.0002
Batch (eff):      16
Max seq length:   1024
Valid SVG rate:   12.9%
Fallbacks:        0/1000
Seed:             42
AI Tooling: Claude (Anthropic) used for code scaffolding and debugging.


In [24]:
# Reset and re-run inference with batch size 4
CONFIG['inference_batch_size'] = 4
torch.cuda.empty_cache()

test_df = pd.read_csv(CONFIG['test_path'])
all_svgs = []
invalid_count = 0
t0 = time.time()

for i in range(0, len(test_df), CONFIG['inference_batch_size']):
    batch = test_df['prompt'].tolist()[i:i + CONFIG['inference_batch_size']]
    svgs = generate_batch(batch)
    all_svgs.extend(svgs)
    invalid_count += sum(1 for s in svgs if s == fallback_svg())
    elapsed = (time.time() - t0) / 60
    print(f'  [{i+len(batch)}/1000] {elapsed:.1f}min | fallbacks: {invalid_count}')

print(f'\nDone! Fallbacks: {invalid_count}/1000')

  [4/1000] 1.3min | fallbacks: 0
  [8/1000] 2.5min | fallbacks: 0
  [12/1000] 3.8min | fallbacks: 0
  [16/1000] 5.0min | fallbacks: 0
  [20/1000] 6.3min | fallbacks: 0
  [24/1000] 7.5min | fallbacks: 0
  [28/1000] 8.8min | fallbacks: 0
  [32/1000] 10.0min | fallbacks: 0
  [36/1000] 11.3min | fallbacks: 0
  [40/1000] 12.6min | fallbacks: 0
  [44/1000] 13.8min | fallbacks: 0
  [48/1000] 15.1min | fallbacks: 0
  [52/1000] 16.4min | fallbacks: 0
  [56/1000] 17.7min | fallbacks: 0
  [60/1000] 19.0min | fallbacks: 0
  [64/1000] 20.3min | fallbacks: 0
  [68/1000] 21.6min | fallbacks: 0
  [72/1000] 22.9min | fallbacks: 0
  [76/1000] 24.2min | fallbacks: 0
  [80/1000] 25.5min | fallbacks: 0
  [84/1000] 26.8min | fallbacks: 0
  [88/1000] 28.1min | fallbacks: 0
  [92/1000] 29.4min | fallbacks: 0
  [96/1000] 30.7min | fallbacks: 0
  [100/1000] 32.0min | fallbacks: 0
  [104/1000] 33.3min | fallbacks: 0
  [108/1000] 34.6min | fallbacks: 0
  [112/1000] 35.9min | fallbacks: 0
  [116/1000] 37.2min | fa

In [25]:
# Save final submission
submission_df = pd.DataFrame({'id': test_df['id'], 'svg': all_svgs})
fallback = fallback_svg()
real_count = sum(1 for s in submission_df['svg'] if s != fallback)
print(f'Real SVGs: {real_count}/1000')
submission_df.to_csv(CONFIG['submission_path'], index=False)
print(f"Saved: {CONFIG['submission_path']}")
print('Submit to Kaggle!')

Real SVGs: 1000/1000
Saved: C:\Users\rahil\Downloads\dl-spring-2026-svg-generation\submission.csv
Submit to Kaggle!


In [3]:
import pandas as pd
import xml.etree.ElementTree as ET

SUBMISSION_PATH = r'C:\Users\rahil\Downloads\dl-spring-2026-svg-generation\submission.csv'

def fallback_svg():
    return ('<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
            '<rect x="0" y="0" width="256" height="256" fill="white"/>'
            '<circle cx="128" cy="128" r="80" fill="#333333"/>'
            '</svg>')

def fix_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return fallback_svg()
    if len(svg_text) <= 8000:
        try:
            ET.fromstring(svg_text)
            return svg_text
        except:
            return fallback_svg()
    try:
        root = ET.fromstring(svg_text)
        root.set('width', '256')
        root.set('height', '256')
        children = list(root)
        while len(ET.tostring(root, encoding='unicode')) > 8000 and children:
            root.remove(children.pop())
        result = ET.tostring(root, encoding='unicode')
        return result if len(result) <= 8000 else fallback_svg()
    except:
        return fallback_svg()

df = pd.read_csv(SUBMISSION_PATH)
print(f'Over 8000 chars: {(df["svg"].str.len() > 8000).sum()}')
df['svg'] = df['svg'].apply(fix_svg)

errors = 0
for i, svg in enumerate(df['svg']):
    try:
        ET.fromstring(svg)
    except:
        df.at[i, 'svg'] = fallback_svg()
        errors += 1

print(f'Parse errors fixed: {errors}')
print(f'Still over 8000: {(df["svg"].str.len() > 8000).sum()}')
df.to_csv(SUBMISSION_PATH, index=False)
print('Saved! Ready to submit.')

Over 8000 chars: 0
Parse errors fixed: 0
Still over 8000: 0
Saved! Ready to submit.
